# 固定新低收敛信息检验：不拟合
只检验既定正向假设，使用2024/2025，排除发现季度但明确历史已被项目复用。完整日历区块重采样、年界标签成熟、四个事前价格状态控制；失败不翻号不换窗口。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from scipy.stats import spearmanr
from src.data.nas_market_data import archive_nas_recorder
from src.data.minute_intake import atomic_json
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/contraction_information_20261014';plan=json.loads((card/'plan.json').read_text());qlib.init(provider_uri=plan['provider_uri'],region='us')
def load(key,artifact):
 spec=plan['sources'][key];p=root/spec['card'];s=json.loads((p/'summary.json').read_text());mp=p/'verification_manifest.json';assert hashlib.sha256(mp.read_bytes()).hexdigest()==spec['manifest_sha256'];entry=next(e for e in json.loads(mp.read_text())['entries'] if e['path'].endswith('/artifacts/'+artifact))
 with R.uri_context(s['recorder_uri']):
  r=R.get_recorder(recorder_id=spec['recorder_id'],experiment_name=p.name);path=Path(r.download_artifact(artifact));assert hashlib.sha256(path.read_bytes()).hexdigest()==entry['sha256'];return r.load_object(artifact)
features=load('descriptors','analysis/features')['old'];prices=load('prices','inputs/frame');grouped=load('labels','inputs/grouped').xs('SOXX',level='instrument');assert grouped.index.equals(prices.index)
frame=pd.DataFrame({'signal_date':prices.signal_date,'decision_at':grouped['meta'].decision_at,'label_available_at':grouped['meta'].label_available_at,'score':features.new_low_price_contraction5.reindex(pd.DatetimeIndex(prices.signal_date)).to_numpy(),'future_soxx_log_return_h5':-grouped['label'].cash_relative_h5},index=prices.index)
r5=grouped['feature'].soxx_ret5;r20=grouped['feature'].soxx_ret20;frame['regime']=np.select([r5.ge(0)&r20.ge(0),r5.ge(0)&r20.lt(0),r5.lt(0)&r20.ge(0)],['uptrend','rebound','pullback'],default='decline')
frame=frame.loc[frame.signal_date.dt.year.isin(plan['evaluation_years'])].copy();frame['feature_available_at']=frame.signal_date+pd.Timedelta(hours=16,minutes=1);assert (frame.feature_available_at<frame.decision_at).all()
end=frame.signal_date.dt.year.map(lambda y:pd.Timestamp(year=y+1,month=1,day=1));mature=frame.label_available_at.notna()&frame.label_available_at.lt(end);frame.loc[~mature,'future_soxx_log_return_h5']=np.nan
assert not frame.signal_date.between('2023-10-01','2023-12-31').any() and not frame.signal_date.between('2026-07-01','2026-09-30').any()
def metrics(data):
 p=data.dropna(subset=['score','future_soxx_log_return_h5']);x=p.score;y=p.future_soxx_log_return_h5
 if len(p)<30 or x.nunique()<2:return {'spearman':np.nan,'within_regime_rank_association':np.nan}
 rx=p.groupby('regime').score.rank();ry=p.groupby('regime').future_soxx_log_return_h5.rank()
 rx=rx-rx.groupby(p.regime).transform('mean');ry=ry-ry.groupby(p.regime).transform('mean')
 return {'spearman':float(spearmanr(x,y).statistic),'within_regime_rank_association':float(rx.corr(ry))}
annual=[]
for year,p in frame.groupby(frame.signal_date.dt.year):annual.append({'year':int(year),'calendar_rows':len(p),'known_mature_rows':int(p[['score','future_soxx_log_return_h5']].notna().all(axis=1).sum()),**metrics(p)})
point=metrics(frame);rng=np.random.default_rng(plan['seed']);n=len(frame);draws=[]
for _ in range(plan['resamples']):
 ix=np.empty(n,dtype=int);ix[0]=rng.integers(n)
 for j in range(1,n):ix[j]=rng.integers(n) if rng.random()<1/plan['mean_block'] else (ix[j-1]+1)%n
 draws.append(metrics(frame.iloc[ix]))
draws=pd.DataFrame(draws);intervals=[]
for metric,estimate in point.items():
 low,high=np.nanquantile(draws[metric],[.025,.975]);intervals.append({'metric':metric,'estimate':estimate,'lower':low,'upper':high,'valid_draws':int(draws[metric].notna().sum())})
passed=all(r['within_regime_rank_association']>0 for r in annual) and next(r for r in intervals if r['metric']=='within_regime_rank_association')['lower']>0
summary={'experiment':card.name,'decision':'descriptive_information_pass_requires_independent_incremental_validation' if passed else 'not_promoted_fixed_contraction_information_failed','primary_metric':'within_regime_rank_association','primary_gate_pass':bool(passed),'new_fits':0,'new_backtests':0,'parameter_searches':0,'objective_achieved':False,'calendar_rows':len(frame),'known_mature_rows':int(frame[['score','future_soxx_log_return_h5']].notna().all(axis=1).sum()),'provider_uri':plan['provider_uri'],'region':'us','as_of':'2026-07-02','qlib_docs':'https://qlib.readthedocs.io/en/latest/component/recorder.html','scope':'Fixed expected-positive score and H5 target, no sign flip or horizon search. Excludes discovery quarters but evaluation history was previously used elsewhere. Coarse four-state rank control is not a full baseline-feature incremental test; target is gross, not net portfolio alpha.','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/frame','analysis/bootstrap']}}
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='fixed_no_fit_contraction_test',uri=str(tracking)):
 rec=R.get_recorder();rec.log_params(new_fits=0,new_backtests=0,expected_sign='positive',horizon=5)
 rec.save_objects(artifact_path='inputs',plan=plan)
 rec.save_objects(artifact_path='analysis',frame=frame,bootstrap=draws,annual=pd.DataFrame(annual),intervals=pd.DataFrame(intervals),summary=summary)
 artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');pd.DataFrame(annual).to_csv(card/'annual_information.csv',index=False);pd.DataFrame(intervals).to_csv(card/'information_intervals.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':plan['provider_uri'],'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False));print(pd.DataFrame(annual).to_string(index=False));print(pd.DataFrame(intervals).to_string(index=False))


# 固定新低收敛假设未获信息支持

本轮0训练、0回测、0参数搜索。只测试既定new_low_price_contraction5：新低发生率的价格指标部分收敛，是否与随后SOXX五个交易区间对数收益正相关。方向、H5、20/5日定义与63日区块均在执行前固定；未测试反号、替代窗口或最优门槛。

评价只取2024和2025，排除提出特征时重点查看的2023Q4、2026Q3。但2024/2025历史已被其他研究反复使用，因此这里只是进一步反证诊断，不是新鲜独立样本外。全部502个原交易位置保留，485行具有已知特征和年界内成熟标签；未知特征、未成熟标签不变成0，也不压缩日历做重采样。

主指标是按事前SOXX过去5/20日方向四状态分别排名、组内中心化后关联。2024为-0.00821（241成熟行），2025为-0.13464（244行）；合并-0.05727，1000次63日平均区块95%区间[-0.19255,0.08421]。普通Spearman两年分别-0.06811/-0.06160，合并-0.06332，区间[-0.18906,0.06664]。有效重采样均1000次，未校正此前全部研究选择。

两个年份均未给出预期正向，合并区间也跨0，固定主门失败。不能将负点估计翻号称为新alpha，也不能根据置信区间重新选期限或状态。四个粗状态控制不能替代完整旧特征的增量比较；即使这里通过，也尚需独立增量及完整净费账户验证。

结论：缺失修复、来源稳定、清晰的损伤/收敛语义，不等于领先收益信息。当前证据没有支持用这项收敛描述减少现金或增加SOXL。该固定正向H5解释关闭；不将其泛化为所有成分信息无效。下跌集中度等其他描述仍是未验证字段，不能当作备选赢家自动替上。

决定：not_promoted_fixed_contraction_information_failed。目标标签为毛SOXX对数收益，不是10万美元Tiered组合收益，未产生任何新的策略净超额，现役策略未改。稳定净超额目标未完成。

[年度固定检验](annual_information.csv) · [区块区间](information_intervals.csv)

Recorder `e8e9a82d82d24527a26011c6ca68a988`。
